# 01 — Dataset Exploration & Quality Analysis

**Purpose:** Explore the Tkllm-darija voice recording dataset collected from
contributors. Analyse audio quality, transcript quality, demographic
distribution, and dialect coverage.

**Datasets sourced:**
- `tkllm-audio` S3/R2 bucket — platform-collected Darija voice recordings
- [DODa](https://github.com/wissamantoun/moroccan-darija-asr) — Moroccan Darija ASR corpus
- [DVoice](https://github.com/MBZUAI-Paris/Darija-Voice) — Darija voice dataset
- [AtlasIA](https://github.com/atlasia/) — Moroccan Arabic NLP resources

**Author:** Brahim Ait Oufkir

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import soundfile as sf
from datasets import load_dataset, DatasetDict
from tqdm.auto import tqdm

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

# S3 / local paths
DATA_ROOT = Path(os.getenv('DATA_ROOT', '../data/samples'))
print(f'Data root: {DATA_ROOT.resolve()}')

## 1. Load Platform Dataset
Load metadata from PostgreSQL or a local Parquet export.

In [ ]:
# Load from local Parquet export (replace with DB query in production)
PARQUET_PATH = DATA_ROOT / 'platform_recordings.parquet'

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
else:
    # Create a synthetic sample for notebook demo
    rng = np.random.default_rng(42)
    n = 500
    df = pd.DataFrame({
        'audio_id':          [f'audio_{i:05d}' for i in range(n)],
        'duration_seconds':  rng.uniform(3, 45, n).round(2),
        'quality_score':     rng.beta(8, 2, n).round(3),
        'region':            rng.choice(['casablanca-settat','marrakech-safi','fes-meknes',
                                         'rabat-sale','souss-massa','tanger-tetouan'], n),
        'age_group':         rng.choice(['18-24','25-34','35-44','45-54','55+'], n,
                                         p=[0.35,0.30,0.20,0.10,0.05]),
        'gender':            rng.choice(['male','female','non-binary'], n, p=[0.52,0.46,0.02]),
        'native_dialect':    rng.choice(['casablanci','marrakchi','fassi','rbati','souss'], n),
        'domain':            rng.choice(['general','banking','healthcare','e-commerce',
                                         'tourism','legal'], n, p=[0.40,0.15,0.15,0.15,0.10,0.05]),
        'language':          rng.choice(['darija','darija-fr','darija-ar','tamazight'], n,
                                         p=[0.55,0.25,0.15,0.05]),
        'snr_db':            rng.normal(25, 8, n).round(1),
        'status':            rng.choice(['approved','rejected','pending'], n, p=[0.75,0.10,0.15]),
        'transcript_wer':    rng.beta(2, 8, n).round(3),
    })
    df['created_at'] = pd.date_range('2024-01-01', periods=n, freq='2h')

print(f'Loaded {len(df):,} recordings')
df.head()

## 2. Basic Statistics

In [ ]:
total_hours = df['duration_seconds'].sum() / 3600
approved = df[df['status'] == 'approved']
approved_hours = approved['duration_seconds'].sum() / 3600

print('=== Dataset Overview ===')
print(f'Total recordings:    {len(df):>8,}')
print(f'Total audio hours:   {total_hours:>8.1f} h')
print(f'Approved recordings: {len(approved):>8,} ({len(approved)/len(df)*100:.1f}%)')
print(f'Approved hours:      {approved_hours:>8.1f} h')
print(f'Avg duration:        {df.duration_seconds.mean():>8.1f} s')
print(f'Avg quality score:   {df.quality_score.mean():>8.3f}')
print(f'Avg SNR:             {df.snr_db.mean():>8.1f} dB')
print(f'Avg transcript WER:  {df.transcript_wer.mean()*100:>8.1f}%')
print(f'Unique regions:      {df.region.nunique():>8}')
print(f'Unique dialects:     {df.native_dialect.nunique():>8}')

## 3. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Tkllm-darija Dataset Distributions', fontsize=16, fontweight='bold')

# Audio duration
axes[0,0].hist(df['duration_seconds'], bins=40, color='#4e79a7', edgecolor='white')
axes[0,0].set_title('Audio Duration Distribution')
axes[0,0].set_xlabel('Duration (seconds)')
axes[0,0].axvline(df['duration_seconds'].median(), color='red', linestyle='--', label='Median')
axes[0,0].legend()

# Quality score
axes[0,1].hist(df['quality_score'], bins=30, color='#59a14f', edgecolor='white')
axes[0,1].set_title('Quality Score Distribution')
axes[0,1].set_xlabel('Quality Score')
axes[0,1].axvline(0.75, color='orange', linestyle='--', label='Min threshold (0.75)')
axes[0,1].axvline(0.90, color='red', linestyle='--', label='Auto-approve (0.90)')
axes[0,1].legend(fontsize=8)

# SNR distribution
axes[0,2].hist(df['snr_db'], bins=30, color='#f28e2b', edgecolor='white')
axes[0,2].set_title('Signal-to-Noise Ratio (SNR)')
axes[0,2].set_xlabel('SNR (dB)')
axes[0,2].axvline(20, color='red', linestyle='--', label='Min acceptable (20 dB)')
axes[0,2].legend()

# Regional distribution
region_counts = df['region'].value_counts()
axes[1,0].barh(region_counts.index, region_counts.values, color='#76b7b2')
axes[1,0].set_title('Recordings by Region')
axes[1,0].set_xlabel('Count')

# Language distribution
lang_counts = df['language'].value_counts()
axes[1,1].pie(lang_counts.values, labels=lang_counts.index,
              autopct='%1.1f%%', colors=sns.color_palette('muted'))
axes[1,1].set_title('Language Distribution')

# Domain distribution
domain_counts = df['domain'].value_counts()
axes[1,2].bar(domain_counts.index, domain_counts.values, color='#edc948')
axes[1,2].set_title('Recordings by Domain')
axes[1,2].set_xlabel('Domain')
axes[1,2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../data/samples/dataset_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dataset_distributions.png')

## 4. Audio Quality Analysis

In [ ]:
def analyse_audio_file(path: str) -> dict:
    """Extract audio quality metrics from a single file."""
    y, sr = librosa.load(path, sr=16000, mono=True)
    duration = len(y) / sr

    # SNR estimate (ratio of signal RMS to noise floor RMS)
    signal_rms = np.sqrt(np.mean(y**2))
    # Estimate noise from quietest 10% of frames
    frame_rms = librosa.feature.rms(y=y, frame_length=512, hop_length=256)[0]
    noise_rms = np.percentile(frame_rms, 10)
    snr_db = 20 * np.log10(signal_rms / (noise_rms + 1e-10))

    # Clipping detection
    clip_ratio = np.mean(np.abs(y) > 0.99)

    # Speech ratio (frames with energy above threshold)
    energy_threshold = np.percentile(frame_rms, 30)
    speech_ratio = np.mean(frame_rms > energy_threshold)

    # Spectral features for quality assessment
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
    zero_crossing_rate = librosa.feature.zero_crossing_rate(y).mean()

    return {
        'duration_s':        round(duration, 2),
        'sample_rate':       sr,
        'snr_db':            round(float(snr_db), 1),
        'clip_ratio':        round(float(clip_ratio), 4),
        'speech_ratio':      round(float(speech_ratio), 3),
        'spectral_centroid': round(float(spectral_centroid), 1),
        'zcr':               round(float(zero_crossing_rate), 4),
        'is_acceptable':     snr_db >= 15 and clip_ratio < 0.01 and duration >= 2,
    }

# Example usage (run on actual audio files)
print('Audio analysis function ready.')
print('Usage: analyse_audio_file("path/to/recording.webm")')

## 5. Demographic Balance Check

In [ ]:
# Check demographic balance for bias detection
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Demographic Balance (CNDP Compliance Check)', fontsize=14)

# Gender
gender_quality = df.groupby('gender')['quality_score'].agg(['mean','count']).round(3)
axes[0].bar(gender_quality.index, gender_quality['mean'],
            color=['#4e79a7','#f28e2b','#59a14f'])
axes[0].set_title('Avg Quality Score by Gender')
axes[0].set_ylim(0, 1)
for i, (idx, row) in enumerate(gender_quality.iterrows()):
    axes[0].text(i, row['mean'] + 0.02, f"n={row['count']}", ha='center', fontsize=9)

# Age group
age_quality = df.groupby('age_group')['quality_score'].mean().sort_index()
axes[1].bar(age_quality.index, age_quality.values, color='#76b7b2')
axes[1].set_title('Avg Quality Score by Age Group')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=30)

# Dialect vs WER correlation
dialect_wer = df.groupby('native_dialect')['transcript_wer'].mean().sort_values()
axes[2].barh(dialect_wer.index, dialect_wer.values * 100, color='#edc948')
axes[2].set_title('Avg Transcript WER by Dialect')
axes[2].set_xlabel('WER (%)')

plt.tight_layout()
plt.show()

# Flag imbalanced demographics
gender_dist = df['gender'].value_counts(normalize=True)
print('\nGender distribution:')
for g, pct in gender_dist.items():
    flag = '⚠️ IMBALANCED' if pct < 0.25 or pct > 0.70 else '✅'
    print(f'  {g:<15} {pct*100:5.1f}%  {flag}')

## 6. Export Summary Report

In [ ]:
report = {
    'total_recordings':     len(df),
    'total_hours':          round(total_hours, 2),
    'approved_hours':       round(approved_hours, 2),
    'avg_quality_score':    round(df.quality_score.mean(), 3),
    'avg_snr_db':           round(df.snr_db.mean(), 1),
    'avg_wer':              round(df.transcript_wer.mean(), 3),
    'unique_regions':       df.region.nunique(),
    'unique_dialects':      df.native_dialect.nunique(),
    'language_distribution': df.language.value_counts(normalize=True).round(3).to_dict(),
    'domain_distribution':   df.domain.value_counts(normalize=True).round(3).to_dict(),
    'gender_distribution':   df.gender.value_counts(normalize=True).round(3).to_dict(),
}

report_path = DATA_ROOT / 'exploration_report.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f'Report saved to {report_path}')
print(json.dumps(report, indent=2, ensure_ascii=False))